# 📊 Corporate Action Processing & Reconciliation System
> **Portfolio Project** — Treasury Operations Analyst | Deutsche Bank

This notebook simulates the **corporate action lifecycle** as handled by custodian banks and prime brokers, including:
- Event capture & classification (dividends, stock splits)
- Position-adjusted cash flow projection with FX & withholding tax
- Ex-date / Record date / Pay date timeline (T+2 settlement convention)
- Settlement reconciliation: expected vs. actual cash received
- Exception & break detection (failed / partial / late settlements)
- Formatted Excel ops report generation (download-ready from Colab)

**Universe:** US blue chips (AAPL, MSFT, GOOGL) + Indonesian equities (BBCA.JK, TLKM.JK, BMRI.JK)

In [ ]:
# ── Install dependencies (Colab-safe) ──────────────────────────────────────
!pip install yfinance openpyxl seaborn -q

In [ ]:
# ==================== IMPORTS & CONFIG ====================
import yfinance as yf
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import seaborn as sns
from datetime import datetime, timedelta
import random
import warnings
warnings.filterwarnings('ignore')

pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', '{:,.4f}'.format)

# Ticker universe: US blue chips + Indonesian equities
TICKERS = ['AAPL', 'MSFT', 'GOOGL', 'BBCA.JK', 'TLKM.JK', 'BMRI.JK']
PERIOD  = '3y'

print('✅ Libraries loaded.')
print(f'   Ticker universe: {TICKERS}')

## Phase 1 — Data Collection & Event Capture

In [ ]:
# ==================== PHASE 1: DATA COLLECTION ====================

def get_stock_data(ticker, period='3y'):
    """Fetch price history and corporate action events for a single ticker."""
    print(f'📥 Fetching data for {ticker}...')
    stock     = yf.Ticker(ticker)
    hist      = stock.history(period=period, auto_adjust=True)
    dividends = stock.dividends
    splits    = stock.splits
    print(f'   ✅ {len(hist)} trading days | {len(dividends)} dividend(s) | {len(splits)} split(s)')
    return stock, hist, dividends, splits


def build_event_table(ticker, hist, dividends, splits):
    """
    Build a structured Corporate Action event table with full date lifecycle:
    Ex-Date → Record Date → Pay Date.
    Record/Pay dates estimated using standard T+1 / T+14 convention where unavailable.
    """
    events = []

    for date, amount in dividends.items():
        ex_date  = date.date()
        rec_date = (date + timedelta(days=1)).date()    # T+1
        pay_date = (date + timedelta(days=14)).date()   # typical ~2-week pay lag
        events.append({
            'Ticker':      ticker,
            'Type':        'Dividend',
            'Ex_Date':     ex_date,
            'Record_Date': rec_date,
            'Pay_Date':    pay_date,
            'Value':       round(amount, 4),
            'Currency':    'USD' if '.JK' not in ticker else 'IDR',
            'Status':      'Confirmed',
        })

    for date, ratio in splits.items():
        events.append({
            'Ticker':      ticker,
            'Type':        'Stock Split',
            'Ex_Date':     date.date(),
            'Record_Date': date.date(),
            'Pay_Date':    date.date(),
            'Value':       ratio,
            'Currency':    'N/A',
            'Status':      'Confirmed',
        })

    return pd.DataFrame(events)


# ── Run for all tickers ────────────────────────────────────────────────────
price_data = {}
all_events = []

for ticker in TICKERS:
    stock, hist, dividends, splits = get_stock_data(ticker, PERIOD)
    price_data[ticker] = hist
    ev = build_event_table(ticker, hist, dividends, splits)
    if not ev.empty:
        all_events.append(ev)

final_events = pd.concat(all_events, ignore_index=True) if all_events else pd.DataFrame()
final_events = final_events.sort_values('Ex_Date', ascending=False).reset_index(drop=True)

print(f'\n📋 Total corporate action events captured: {len(final_events)}')
print(final_events.groupby(['Ticker', 'Type']).size().to_string())
final_events.head(10)

## Phase 2 — Cash Flow Projection by Holdings Position

In [ ]:
# ==================== PHASE 2: CASH FLOW PROJECTION ====================
# Simulates a portfolio holdings book and projects expected dividend cash inflows.

# Simulated holdings (shares held per ticker)
HOLDINGS = {
    'AAPL':    10_000,
    'MSFT':     5_000,
    'GOOGL':    2_500,
    'BBCA.JK': 50_000,
    'TLKM.JK': 80_000,
    'BMRI.JK': 60_000,
}

# FX rates to USD (approximate)
FX = {'USD': 1.0, 'IDR': 1 / 16_000}

def project_cash_flows(events_df, holdings, fx):
    """Project expected net cash inflows from dividend events based on holdings."""
    rows = []
    dividend_events = events_df[events_df['Type'] == 'Dividend'].copy()

    for _, ev in dividend_events.iterrows():
        ticker    = ev['Ticker']
        shares    = holdings.get(ticker, 0)
        ccy       = ev['Currency']
        gross     = round(ev['Value'] * shares, 2)
        gross_usd = round(gross * fx.get(ccy, 1.0), 2)
        wht       = round(gross_usd * 0.15, 2)   # 15% withholding tax
        net_usd   = round(gross_usd - wht, 2)

        rows.append({
            'Ticker':        ticker,
            'Pay_Date':      ev['Pay_Date'],
            'Shares_Held':   shares,
            'DPS':           ev['Value'],
            'Currency':      ccy,
            'Gross_CCY':     gross,
            'Gross_USD':     gross_usd,
            'WHT_15pct_USD': wht,
            'Net_USD':       net_usd,
        })

    cf = pd.DataFrame(rows).sort_values('Pay_Date', ascending=False)
    return cf


cash_flows = project_cash_flows(final_events, HOLDINGS, FX)

print(f'💰 Cash Flow Projection — {len(cash_flows)} dividend payments')
print(f'   Total Gross USD : ${cash_flows["Gross_USD"].sum():>12,.2f}')
print(f'   Total WHT       : ${cash_flows["WHT_15pct_USD"].sum():>12,.2f}')
print(f'   Total Net USD   : ${cash_flows["Net_USD"].sum():>12,.2f}')
cash_flows.head(10)

## Phase 3 — Settlement Reconciliation & Exception Detection

In [ ]:
# ==================== PHASE 3: SETTLEMENT RECONCILIATION ====================
# Simulates the ops reconciliation workflow:
#   Expected cash (from CA system) vs. Actual cash (from custodian/bank statement)
#   Flags breaks: missing, short, or late payments.

random.seed(42)

def simulate_actual_settlements(cash_flows_df):
    """
    Simulate custodian-received cash vs. expected amounts.
    Introduces realistic discrepancies:
      - 70% settle in full  (Matched)
      - 15% receive a short payment (Partial)
      - 10% are delayed 1-5 business days (Late)
      -  5% are missing entirely (Failed)
    """
    scenarios = ['Matched']*7 + ['Partial']*2 + ['Late']*2 + ['Failed']
    rows = []
    for _, row in cash_flows_df.iterrows():
        scenario = random.choice(scenarios)
        expected = row['Net_USD']

        if scenario == 'Matched':
            actual    = expected
            actual_dt = row['Pay_Date']
        elif scenario == 'Partial':
            actual    = round(expected * random.uniform(0.80, 0.99), 2)
            actual_dt = row['Pay_Date']
        elif scenario == 'Late':
            actual    = expected
            delay     = random.randint(1, 5)
            actual_dt = (pd.to_datetime(row['Pay_Date']) + timedelta(days=delay)).date()
        else:  # Failed
            actual    = 0.0
            actual_dt = None

        rows.append({
            'Ticker':            row['Ticker'],
            'Pay_Date_Expected': row['Pay_Date'],
            'Pay_Date_Actual':   actual_dt,
            'Expected_Net_USD':  expected,
            'Actual_USD':        actual,
            'Break_USD':         round(actual - expected, 2),
            'Status':            scenario,
            'Action_Required':   'YES' if scenario != 'Matched' else 'No',
        })

    return pd.DataFrame(rows)


recon = simulate_actual_settlements(cash_flows)

print('=== RECONCILIATION SUMMARY ===')
summary = recon.groupby('Status').agg(
    Count           = ('Break_USD', 'count'),
    Total_Break_USD = ('Break_USD', 'sum'),
).round(2)
print(summary.to_string())
print(f'\n⚠️  Items Requiring Action  : {(recon["Action_Required"]=="YES").sum()}')
print(f'   Total Break Exposure USD : ${recon["Break_USD"].abs().sum():,.2f}')

exceptions = recon[recon['Action_Required'] == 'YES'].sort_values('Break_USD')
print(f'\n🚨 EXCEPTION REPORT ({len(exceptions)} items):')
exceptions

## Phase 4 — Event Study: Price Impact Analysis

In [ ]:
# ==================== PHASE 4: EVENT STUDY ====================
# Measures cumulative return around each corporate action event (±10 trading days post).

def event_study(hist, event_date, window=30):
    """Compute cumulative return relative to price on event day (day 0)."""
    event_date = pd.to_datetime(event_date).tz_localize(None)
    hist.index = hist.index.tz_localize(None)
    start = event_date - timedelta(days=window + 10)
    end   = event_date + timedelta(days=window + 10)
    w     = hist.loc[start:end, ['Close']].copy()
    if len(w) < 5:
        return None
    pre = w[w.index <= event_date]
    if pre.empty:
        return None
    anchor      = pre['Close'].iloc[-1]
    w['Days']   = (w.index - event_date).days
    w['Return'] = (w['Close'] / anchor - 1) * 100
    return w[w['Days'].between(-window, window)]


def post_event_return(hist, event_date, days_after=10):
    """Return the N-day post-event cumulative return."""
    w = event_study(hist, event_date, window=days_after + 5)
    if w is None or len(w) < 3:
        return None
    post = w[w['Days'] >= 0]
    return round(post['Return'].iloc[-1], 2) if len(post) >= 2 else None


analysis = []
for _, ev in final_events.iterrows():
    ticker = ev['Ticker']
    hist   = price_data.get(ticker)
    if hist is None or hist.empty:
        continue
    ret = post_event_return(hist, ev['Ex_Date'], days_after=10)
    if ret is None:
        continue
    analysis.append({
        'Ticker':         ticker,
        'Type':           ev['Type'],
        'Ex_Date':        ev['Ex_Date'],
        'Value':          ev['Value'],
        'Return_10d (%)': ret,
        'Signal':         '🟢 Up' if ret > 0 else '🔴 Down',
    })

df_analysis = pd.DataFrame(analysis)

print(f'✅ Event study complete — {len(df_analysis)} events analysed.')
print('\n=== RETURN SUMMARY BY TICKER ===')
print(df_analysis.groupby('Ticker')['Return_10d (%)'].agg(['count','mean','min','max']).round(2).to_string())
df_analysis.head(10)

## Phase 5 — Analytics Dashboard (4-panel Chart)

In [ ]:
# ==================== PHASE 5: VISUALIZATIONS ====================

fig, axes = plt.subplots(2, 2, figsize=(16, 10))
fig.suptitle('Corporate Action Analytics Dashboard', fontsize=16, fontweight='bold')

# Plot 1: Event count by ticker & type
ax1 = axes[0, 0]
ct = final_events.groupby(['Ticker', 'Type']).size().unstack(fill_value=0)
ct.plot(kind='bar', ax=ax1, colormap='Set2', edgecolor='black', linewidth=0.5)
ax1.set_title('Corporate Action Events by Ticker')
ax1.set_xlabel('Ticker')
ax1.set_ylabel('Number of Events')
ax1.legend(title='Type')
ax1.tick_params(axis='x', rotation=30)

# Plot 2: 10-day return distribution by CA type
ax2 = axes[0, 1]
if not df_analysis.empty:
    sns.boxplot(x='Type', y='Return_10d (%)', data=df_analysis, ax=ax2, palette='pastel')
    ax2.axhline(0, color='red', linestyle='--', linewidth=1, label='Break-even')
    ax2.set_title('10-Day Post-Event Return by CA Type')
    ax2.set_xlabel('Corporate Action Type')
    ax2.set_ylabel('Return (%)')
    ax2.legend()

# Plot 3: Reconciliation status breakdown
ax3 = axes[1, 0]
status_counts = recon['Status'].value_counts()
colors_map = {'Matched': '#2ecc71', 'Partial': '#f39c12', 'Late': '#3498db', 'Failed': '#e74c3c'}
ax3.pie(
    status_counts.values,
    labels=status_counts.index,
    colors=[colors_map.get(s, 'grey') for s in status_counts.index],
    autopct='%1.1f%%', startangle=90,
    wedgeprops={'edgecolor': 'white', 'linewidth': 1.5}
)
ax3.set_title('Settlement Reconciliation Status')

# Plot 4: Monthly net cash flow projection
ax4 = axes[1, 1]
cf_monthly = cash_flows.copy()
cf_monthly['Pay_Date'] = pd.to_datetime(cf_monthly['Pay_Date'])
cf_monthly = cf_monthly.set_index('Pay_Date')['Net_USD'].resample('ME').sum().reset_index()
ax4.bar(cf_monthly['Pay_Date'], cf_monthly['Net_USD'] / 1e3,
        color='steelblue', edgecolor='black', linewidth=0.4)
ax4.set_title('Monthly Projected Net Cash Inflow (USD)')
ax4.set_xlabel('Month')
ax4.set_ylabel('Net Cash (USD thousands)')
ax4.xaxis.set_major_formatter(mdates.DateFormatter('%b %Y'))
ax4.xaxis.set_major_locator(mdates.MonthLocator(interval=3))
plt.setp(ax4.xaxis.get_majorticklabels(), rotation=30, ha='right')

plt.tight_layout()
plt.savefig('ca_dashboard.png', dpi=150, bbox_inches='tight')
plt.show()
print('📊 Chart saved as ca_dashboard.png')

## Phase 6 — Excel Ops Report (Download-ready)

In [ ]:
# ==================== PHASE 6: EXCEL OPS REPORT ====================
# Generates a formatted multi-sheet Excel report ready to send to ops/compliance.

from openpyxl import load_workbook
from openpyxl.styles import Font, PatternFill, Alignment, Border, Side
from openpyxl.utils import get_column_letter

REPORT_FILE = 'CorpAct_OpsReport.xlsx'

with pd.ExcelWriter(REPORT_FILE, engine='openpyxl') as writer:
    final_events.to_excel(writer, sheet_name='CA_Events',          index=False)
    cash_flows.to_excel(writer,   sheet_name='Cash_Flow_Proj',     index=False)
    recon.to_excel(writer,        sheet_name='Recon_&_Exceptions',  index=False)
    df_analysis.to_excel(writer,  sheet_name='Event_Study',         index=False)
    exceptions.to_excel(writer,   sheet_name='Action_Required',     index=False)

# Post-process: add formatting
wb          = load_workbook(REPORT_FILE)
HEADER_FILL = PatternFill('solid', fgColor='003366')  # navy
ALERT_FILL  = PatternFill('solid', fgColor='FFD7D7')  # soft red
HEADER_FONT = Font(color='FFFFFF', bold=True, size=10)
THIN        = Side(style='thin', color='CCCCCC')
BORDER      = Border(left=THIN, right=THIN, top=THIN, bottom=THIN)

for sheet_name in wb.sheetnames:
    ws = wb[sheet_name]
    for cell in ws[1]:
        cell.fill      = HEADER_FILL
        cell.font      = HEADER_FONT
        cell.alignment = Alignment(horizontal='center', vertical='center')
        cell.border    = BORDER
    for col in ws.columns:
        max_len = max((len(str(c.value)) if c.value else 0) for c in col)
        ws.column_dimensions[get_column_letter(col[0].column)].width = min(max_len + 4, 30)
    # Highlight exception rows in reconciliation sheet
    if sheet_name == 'Recon_&_Exceptions':
        for row in ws.iter_rows(min_row=2):
            action_cell = [c for c in row if c.column == 8]
            if action_cell and action_cell[0].value == 'YES':
                for cell in row:
                    cell.fill = ALERT_FILL

wb.save(REPORT_FILE)
print(f'✅ Excel ops report saved: {REPORT_FILE}')
print('   Sheets: CA_Events | Cash_Flow_Proj | Recon_&_Exceptions | Event_Study | Action_Required')

# Auto-download in Colab
try:
    from google.colab import files
    files.download(REPORT_FILE)
    print('📥 Download triggered!')
except ImportError:
    print('   (Not in Colab — file is in your working directory)')

## Summary

| Module | What It Demonstrates |
|---|---|
| Phase 1 — Event Capture | CA lifecycle: ex-date, record date, pay date (T+2 convention) |
| Phase 2 — Cash Flow Projection | Position-adjusted dividend income with FX conversion & 15% WHT |
| Phase 3 — Reconciliation | Expected vs. actual settlement; break/exception flagging |
| Phase 4 — Event Study | Quantitative price impact analysis (abnormal return ±10d) |
| Phase 5 — Visualizations | 4-panel dashboard: events, return dist, recon status, cash flow |
| Phase 6 — Excel Report | Formatted multi-sheet ops report (navy header, exception highlight) |

---
*Built for Treasury Operations Analyst portfolio — Deutsche Bank application*